# Lesson 11 · Simulation: does empirical Bayes actually work?

**Source:** *Introduction to Empirical Bayes: Examples from Baseball Statistics*, David Robinson (2017), Chapter 12 (pp. 115–129)

**Question:** We never get to see a player's true batting average, so how do we know shrinkage helps? Build a world where we *do* know the truth, and check.

**Goal:** Simulate batting records from a known prior, then measure how well shrinkage, credible intervals, q-values and beta-binomial regression recover the truth.

**Contents**

1. The generative model
2. Simulate one world
3. Shrinkage: accuracy and bias
4. Credible intervals: are they calibrated?
5. FDR control: does q < 0.1 mean 10%?
6. Beta-binomial regression: parameter recovery
7. Exercises

**Tags:** `simulation` `bias-variance` `mean-squared-error` `calibration` `coverage` `fdr-control` `parameter-recovery`

**Before you start:** Lesson 10 (`my_eb.py`). If you skipped it, use your functions from Lessons 02–06 instead.

*How this notebook works:* each **Your turn** prompt is followed by an empty cell for your code. **Check** boxes give values to compare against; they come from the book, and your numbers can differ slightly because the Lahman data now runs through more recent seasons. **Reflect** prompts are for short written answers.

In [ ]:
# --- Setup: run this cell first ------------------------------------------
# Windows + conda: put the environment's DLL folders on PATH, so plotting
# can't crash the kernel when Jupyter wasn't started from the activated env.
# (If ENVIRONMENT_NOTES.md in the Cardprice folder has a PATH-fix block,
# you can paste that here instead.)
import os, sys
for sub in ["Library\\mingw-w64\\bin", "Library\\usr\\bin", "Library\\bin", "Scripts", "bin"]:
    p = os.path.join(sys.prefix, sub)
    if os.path.isdir(p) and p not in os.environ["PATH"]:
        os.environ["PATH"] = p + os.pathsep + os.environ["PATH"]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy import stats, optimize, special

import eb_data   # helpers in this folder: loading the Lahman data, saving results

rng = np.random.default_rng(2017)
pd.set_option("display.precision", 4)
plt.rcParams["figure.figsize"] = (7, 4)
print(f"numpy {np.__version__} | pandas {pd.__version__} | scipy {scipy.__version__}")

%load_ext autoreload
%autoreload 2
try:
    import my_eb   # your toolkit from Lesson 10
except ImportError:
    print("my_eb.py not found: finish Lesson 10, or define the functions you need below.")

career = eb_data.career(pitcher_rule=3)[["playerID", "H", "AB"]]
print(f"{len(career):,} players")

## 1. The generative model

$$p_i \sim \text{Beta}(\alpha_0, \beta_0), \qquad H_i \sim \text{Binomial}(AB_i, p_i).$$

- $\alpha_0, \beta_0$: estimate them from the real data, so the simulated world looks realistic.
- $p_i$: simulate. These are the true averages we normally never see.
- $AB_i$: reuse the real values. We aren't trying to estimate them.

**Your turn.** Fit the prior to **all** players this time (no AB filter). Plot the distribution of AB (log x) to see why we reuse it rather than simulate it.

In [ ]:
# Your code here

In [ ]:
# Your code here

> **Check:** the book got α₀ ≈ 72.1, β₀ ≈ 215.1.

## 2. Simulate one world

**Your turn.** With a fixed seed, add a true `p` (beta draws) and a simulated `H` (binomial draws using the real AB) to a copy of `career`. Then run your empirical Bayes estimation on the simulated data.

In [ ]:
# Your code here

## 3. Shrinkage: accuracy and bias

**Your turn.** Two panels: raw estimate vs true p, and shrunken estimate vs true p (AB ≥ 10, coloured by AB). Add the y = x line and a fitted regression line to each.

In [ ]:
# Your code here

**Your turn.** Compute the overall mean squared error of raw and shrunken estimates.

In [ ]:
# Your code here

> **Check:** raw MSE ≈ 0.015; shrunken ≈ 0.00035.

**Your turn.** Bin players by AB (`10 ** np.round(np.log10(AB))`). Within each bin compute the MSE and the *slope* of estimate on true p (a slope below 1 means bias toward the middle). Plot both against AB, for raw and shrunken.

In [ ]:
# Your code here

In [ ]:
# Your code here

**Reflect.** Shrinkage adds bias and removes variance. Where is that trade most worthwhile, and where does it hardly matter?

*Your answer:*

## 4. Credible intervals: are they calibrated?

**Your turn.** What fraction of 95% credible intervals contain the true p? Then plot 20 random players' intervals with the true p as a red dot.

In [ ]:
# Your code here

In [ ]:
# Your code here

> **Check:** coverage ≈ 0.95.

**Your turn.** Repeat for credible levels from 0.50 to 0.98 and plot coverage against level, with the line y = x. Print progress as you loop.

In [ ]:
# Your code here

## 5. FDR control: does q < 0.1 mean 10%?

**Your turn.** Run your proportion test against .300 on the simulated data. Admit players with q ≤ 0.1. What fraction of them truly have p < .300?

In [ ]:
# Your code here

> **Check:** about 0.09–0.10.

**Your turn.** For every q-value threshold, plot the *true* false discovery proportion (cumulative mean of `p < 0.3`, sorted by q) against the threshold.

In [ ]:
# Your code here

## 6. Beta-binomial regression: parameter recovery

Now make the world more realistic: let the prior mean depend on log10(AB), as in Lesson 06. Note that `ebbr` uses a **logit** link here, so its coefficients aren't on the same scale as Lesson 06's.

**Your turn.** Fit the regression prior to the real data. Use the fitted coefficients as the *true* hyperparameters: simulate p from each player's own prior, then H.

In [ ]:
# Your code here

> **Check:** with a logit link the book got μ intercept ≈ −1.688, log10(AB) slope ≈ 0.192, log σ ≈ −6.30.

**Your turn.** Refit the regression on the simulated data. Do you recover the true coefficients (compare with their standard errors)?

In [ ]:
# Your code here

**Your turn.** Compare shrunken estimates against true p using (a) a single flat prior and (b) the AB-dependent prior.

In [ ]:
# Your code here

**Reflect.** **Card connection.** Cardprice notebook 07b has a `USE_SIMULATED` switch for exactly this kind of parameter-recovery check. What did it tell you about that model? What's one check from this lesson you could add there?

*Your answer:*

## 7. Exercises

**Your turn.** Fit the prior by the method of moments instead of MLE and repeat the MSE and coverage checks. Is it noticeably worse?

In [ ]:
# Your code here

## References

- *Introduction to Empirical Bayes: Examples from Baseball Statistics*, David Robinson (2017), Chapter 12.
- Morris, White & Crowther (2019), "Using simulation studies to evaluate statistical methods", *Statistics in Medicine* 38(11).
- Talts et al. (2018), "Validating Bayesian inference algorithms with simulation-based calibration", arXiv:1804.06788.